# 04c. 유전자 줄이기 실험 (E 조합 + chi2)

- 담당: **④ 경량화·통합** (이 파일은 담당자만 수정)
- 목적: E 조합(0/1 + 개수 + 변이 종류 + 아미노산 쌍 + 끊김 표시, 9,174열)에서 chi2로 열을 줄여도 성능이 유지되는지 확인
- 모델: 로지스틱 회귀 (C=0.1, class_weight=balanced, max_iter=3000), 팀 그룹 K-fold
- 누수 방지: chi2 선택은 Pipeline 안에서 fold의 학습 부분에서만 계산해요. E 피처 계산은 한 환자 안에서만 하는 계산이라 미리 한 번 계산해 둬요.
- 기록 파일: `experiments/log_P_light.csv` (`1109_E_chi2_k○○_gkf`)
- 예측 파일: `experiments/preds/P_light__logreg_E_chi2_k500_1109__*`
- ⚠️ Run All 하면 "3. 열 수별 교차검증"이 다시 기록돼 기록표에 같은 줄이 또 생겨요.

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [3]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/data


## 1. 도구·데이터 불러오기

In [4]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.features import (mutation_count, mutation_type_counts,
                          amino_acid_pair_counts, gene_truncation_flags)
from src.validation import get_folds, run_cv
from src.evaluation import log_result, save_predictions

train, test = load_data(DATA_DIR)   # test는 마지막 예측에만 사용
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                # 팀 공통 그룹 K-fold
print("train", train.shape, "| 암종 수", len(le.classes_), "| fold 수", len(folds))

train (6201, 4386) | 암종 수 26 | fold 수 5


## 2. E 피처 미리 계산
② 02_features의 make_features_E와 같은 계산이에요. 한 환자 안에서만 하는 계산이라(통계·선택 없음) 미리 한 번 계산해도 누수가 없어요.

In [5]:
def make_features_E(X):
    """② E 조합 피처 (02_features의 make_features_E와 같은 계산). 한 환자 안에서만 계산."""
    return pd.concat([binarize_mutations(X),
                      np.log1p(mutation_count(X)),
                      np.log1p(mutation_type_counts(X)),
                      np.log1p(amino_acid_pair_counts(X)),
                      gene_truncation_flags(X)], axis=1)

X_E = make_features_E(X).astype(np.float32)   # 약 1분
print("E 피처:", X_E.shape)

E 피처: (6201, 9174)


## 3. 열 수별 교차검증 — 2000 / 1000 / 500 / 300 / 100
chi2로 열을 k개만 골라 같은 로지스틱 회귀로 비교해요. 고르는 기준은 fold의 학습 부분에서만 계산해요.

In [6]:
def to_sparse(A):
    """0이 대부분인 표를 가볍게 바꿔 계산 속도를 높임."""
    return csr_matrix(np.asarray(A, dtype=np.float32))

LR_PARAMS = dict(C=0.1, class_weight="balanced", max_iter=3000)   # ② E 실험과 같은 설정

def make_panel_pipe(k):
    return Pipeline([
        ("select", SelectKBest(chi2, k=k)),            # fold 학습 부분에서만 k열 선택
        ("to_sparse", FunctionTransformer(to_sparse)),
        ("model", LogisticRegression(**LR_PARAMS)),
    ])

PANEL_SIZES = [2000, 1000, 500, 300, 100]
panel_results = {}
for k in PANEL_SIZES:
    print(f"\n=== k = {k} ===")
    cv_k = run_cv(make_panel_pipe(k), X_E, y, folds=folds)
    panel_results[k] = cv_k
    log_result(owner="P_light", notebook="04c_gene_panel.ipynb",
               experiment=f"1109_E_chi2_k{k}_gkf",
               features=f"E(0/1+개수+변이 종류+아미노산 쌍+끊김) → SelectKBest(chi2, k={k})",
               model="LogisticRegression", params={**LR_PARAMS, "k": k}, cv=cv_k,
               notes="유전자 줄이기, 그룹 K-fold")


=== k = 2000 ===
fold 1: Macro F1 = 0.4933
fold 2: Macro F1 = 0.4798
fold 3: Macro F1 = 0.4690
fold 4: Macro F1 = 0.5043
fold 5: Macro F1 = 0.5009
CV Macro F1 = 0.4895 (std 0.0132)
기록: log_P_light.csv ← 1109_E_chi2_k2000_gkf (CV 0.4895)

=== k = 1000 ===
fold 1: Macro F1 = 0.4714
fold 2: Macro F1 = 0.4854
fold 3: Macro F1 = 0.4543
fold 4: Macro F1 = 0.4978
fold 5: Macro F1 = 0.4927
CV Macro F1 = 0.4803 (std 0.0158)
기록: log_P_light.csv ← 1109_E_chi2_k1000_gkf (CV 0.4803)

=== k = 500 ===
fold 1: Macro F1 = 0.4505
fold 2: Macro F1 = 0.4761
fold 3: Macro F1 = 0.4508
fold 4: Macro F1 = 0.4851
fold 5: Macro F1 = 0.4846
CV Macro F1 = 0.4694 (std 0.0156)
기록: log_P_light.csv ← 1109_E_chi2_k500_gkf (CV 0.4694)

=== k = 300 ===
fold 1: Macro F1 = 0.4471
fold 2: Macro F1 = 0.4397
fold 3: Macro F1 = 0.4289
fold 4: Macro F1 = 0.4744
fold 5: Macro F1 = 0.4655
CV Macro F1 = 0.4511 (std 0.0167)
기록: log_P_light.csv ← 1109_E_chi2_k300_gkf (CV 0.4511)

=== k = 100 ===
fold 1: Macro F1 = 0.4131
fold 2: M

## 4. 결과 표 — 열 수에 따른 성능
비교 기준: E 전체 9,174열 = ② E_aa_pairs_truncation_group_lr (CV 0.4919)

In [7]:
FULL_CV = 0.4919   # ② E 전체 (log_H_data.csv의 E_aa_pairs_truncation_group_lr)
panel_table = pd.DataFrame(
    [{"열 수": "전체 9,174", "CV Macro F1": FULL_CV, "std": None, "전체 대비(%)": 100.0}]
    + [{"열 수": k, "CV Macro F1": round(r["mean"], 4), "std": round(r["std"], 4),
        "전체 대비(%)": round(r["mean"] / FULL_CV * 100, 1)} for k, r in panel_results.items()]
)
panel_table

,열 수,CV Macro F1,std,전체 대비(%)
0,"전체 9,174",0.4919,NaN,100.0
1,2000,0.4895,0.0132,99.5
2,1000,0.4803,0.0158,97.6
3,500,0.4694,0.0156,95.4
4,300,0.4511,0.0167,91.7
5,100,0.4209,0.0094,85.6


## 5. 500열 모델 예측 파일 저장
- 가설: 열을 줄이면 같은 fold에 맞춰지는 정도가 줄어, CV와 실제 성능(홀드아웃) 차이가 작아진다
- 비교 대상: 같은 로지스틱 회귀의 E 전체 9,174열 (CV 0.4919)
- 판단 기준: "CV − 홀드아웃" 차이가 500열이 E 전체보다 0.015 이상 작으면 효과 있음 (0.015 = LR E CV fold 표준편차 0.014 기준, 결과 확인 전 확정)
- 확인 시점: 팀 홀드아웃 분할 함수(③ 담당)가 생긴 뒤

In [8]:
K_SAVE = 500
cv_save = panel_results[K_SAVE]
final_k = make_panel_pipe(K_SAVE).fit(X_E, y)                  # train 전체로 학습
test_E = make_features_E(test[X.columns]).astype(np.float32)   # test는 예측에만 사용 (한 환자 안 계산)
test_proba_k = final_k.predict_proba(test_E)

save_predictions(owner="P_light", name="logreg_E_chi2_k500_1109",
                 oof_proba=cv_save["oof_proba"], test_proba=test_proba_k,
                 classes=le.classes_, cv=cv_save,
                 notes="E 피처에서 chi2로 500열 선택, 그룹 K-fold")

예측 저장: P_light__logreg_E_chi2_k500_1109 (CV 0.4694) → experiments/preds/


## 6. 500열 구성 — 피처 종류와 알려진 암 유전자 포함 여부

In [9]:
selected = X_E.columns[final_k.named_steps["select"].get_support()]
kind = np.select([selected.str.startswith("trunc_"), selected.str.startswith("aa_"),
                  selected.str.startswith("n_")],
                 ["끊김 표시", "아미노산 쌍", "개수"], default="0/1 변이")
print(pd.Series(kind).value_counts())

genes = {c.replace("trunc_", "") for c in selected if not c.startswith(("aa_", "n_"))}
print("관련 유전자 수:", len(genes))

KNOWN_GENES = ["TP53", "PIK3CA", "PTEN", "APC", "BRAF", "IDH1", "VHL", "NRAS",
               "EGFR", "CDH1", "GATA3", "CTNNB1", "ATRX", "NF1", "RB1", "KRAS"]
print("포함된 알려진 암 유전자:", [g for g in KNOWN_GENES if g in genes])
print("데이터에 없는 유전자:", [g for g in KNOWN_GENES if g not in X.columns])

0/1 변이    354
아미노산 쌍    118
끊김 표시      22
개수          6
Name: count, dtype: int64
관련 유전자 수: 358
포함된 알려진 암 유전자: ['TP53', 'PIK3CA', 'PTEN', 'APC', 'BRAF', 'IDH1', 'VHL', 'EGFR', 'CDH1', 'GATA3', 'CTNNB1', 'ATRX', 'NF1']
데이터에 없는 유전자: ['NRAS', 'KRAS']
